# 04 — Exploratory Analysis

Ad-hoc queries against the Silver and Gold layers. Demonstrates MongoDB aggregation pipelines via pymongo and DataFrame exploration via PySpark.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from pymongo import MongoClient
from core.config.settings import settings
import pandas as pd

client = MongoClient(settings.mongo_uri())
print("MongoDB connected.")

## Audit Summary — Recent Pipeline Runs

In [ ]:
audit_db = client[settings.MONGO_DB_AUDIT]

recent_runs = list(
    audit_db["pipeline_runs"].find(
        {},
        {"execution_id": 1, "pipeline_name": 1, "status": 1,
         "duration_seconds": 1, "output_summary": 1, "_id": 0}
    ).sort("start_time", -1).limit(10)
)

df_runs = pd.DataFrame(recent_runs)
display(df_runs)

## Quarantine Summary — Rejection Reasons

In [ ]:
pipeline = [
    {"$group": {"_id": "$reason", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]

quarantine_summary = list(audit_db["quarantine"].aggregate(pipeline))
df_q = pd.DataFrame(quarantine_summary).rename(columns={"_id": "rejection_reason"})
display(df_q)

## Gold KPIs — Top 10 Zones by Revenue

In [ ]:
gold_db = client[settings.MONGO_DB_GOLD]

top_zones = list(
    gold_db["kpi_revenue_zone"]
    .find({}, {"zone_name": 1, "borough": 1, "trip_count": 1, "total_revenue": 1, "_id": 0})
    .sort("total_revenue", -1)
    .limit(10)
)

df_zones = pd.DataFrame(top_zones)
display(df_zones)

## Gold KPIs — Hourly Volume Trend

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

hourly = list(
    gold_db["kpi_hourly_volume"]
    .aggregate([
        {"$group": {"_id": "$pickup_hour", "total_trips": {"$sum": "$trip_count"}}},
        {"$sort": {"_id": 1}}
    ])
)

df_hourly = pd.DataFrame(hourly).rename(columns={"_id": "hour"})

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(df_hourly["hour"], df_hourly["total_trips"], color="#4A90D9", edgecolor="white")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Total Trips")
ax.set_title("NYC TLC — Trip Volume by Hour of Day")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()